SIB - P5
A Jupyter Notebook with the backward propagation exercises (DenseLayer, activations and Dropout).

In [1]:
import numpy as np

from si.neural_networks.layers import DenseLayer, Dropout
from si.neural_networks.activation import ReLUActivation, SigmoidActivation, TanhActivation, SoftmaxActivation

## Exercício 1 – um passo de treino à mão (slides 10-11)
Rede 2 → 3 → 1 sem ativações, x = [1, 2], y = 1, α = 0.1, E = (y − ŷ)².

In [2]:
x = np.array([[1, 2]])
y = 1
alpha = 0.1

W1 = np.array([[0.1, 0.2, -0.1],
               [0.3, -0.2, 0.4]])
b1 = np.array([[0.1, 0, -0.1]])
W2 = np.array([[0.2], [-0.3], [0.1]])
b2 = np.array([[0.0]])

# 1. forward e loss
h = x @ W1 + b1
y_hat = h @ W2 + b2
loss = (y - y_hat) ** 2
print("h:", h, "| ŷ:", y_hat, "| loss:", loss)

# 2. dE/dŷ
dE_dy = 2 * (y_hat - y)
print("dE/dŷ:", dE_dy)

# 3. gradientes da 2.ª camada e erro passado para trás
dE_dW2 = h.T @ dE_dy
dE_db2 = np.sum(dE_dy, axis=0, keepdims=True)
dE_dh = dE_dy @ W2.T
print("dE/dW2:", dE_dW2.ravel(), "| dE/db2:", dE_db2, "| dE/dh:", dE_dh)

# 4. atualizar W2 e b2
W2_new = W2 - alpha * dE_dW2
b2_new = b2 - alpha * dE_db2
print("W2 novo:", W2_new.ravel(), "| b2 novo:", b2_new)

# 5. atualizar W1 e b1 e fazer o forward outra vez
W1_new = W1 - alpha * x.T @ dE_dh
b1_new = b1 - alpha * np.sum(dE_dh, axis=0, keepdims=True)
print("W1 novo:", W1_new)
y_hat_new = (x @ W1_new + b1_new) @ W2_new + b2_new
print("novo ŷ:", y_hat_new, "| nova loss:", (y - y_hat_new) ** 2)

h: [[ 0.8 -0.2  0.6]] | ŷ: [[0.28]] | loss: [[0.5184]]
dE/dŷ: [[-1.44]]
dE/dW2: [-1.152  0.288 -0.864] | dE/db2: [[-1.44]] | dE/dh: [[-0.288  0.432 -0.144]]
W2 novo: [ 0.3152 -0.3288  0.1864] | b2 novo: [[0.144]]
W1 novo: [[ 0.1288  0.1568 -0.0856]
 [ 0.3576 -0.2864  0.4288]]
novo ŷ: [[0.72955648]] | nova loss: [[0.0731397]]


O mesmo passo, agora com a `DenseLayer.backward_propagation` e um optimizer simples (gradient descent com α = 0.1):

In [3]:
class GD:
    def __init__(self, alpha):
        self.alpha = alpha
    def update(self, w, grad):
        return w - self.alpha * grad

l1 = DenseLayer(3, (2,)).initialize(GD(0.1))
l1.weights, l1.biases = W1.copy(), b1.copy()
l2 = DenseLayer(1, (3,)).initialize(GD(0.1))
l2.weights, l2.biases = W2.copy(), b2.copy()

y_hat = l2.forward_propagation(l1.forward_propagation(x))
print("loss antes:", ((y - y_hat) ** 2).item())
l1.backward_propagation(l2.backward_propagation(2 * (y_hat - y)))
y_hat = l2.forward_propagation(l1.forward_propagation(x))
print("loss depois:", ((y - y_hat) ** 2).item())

loss antes: 0.5184
loss depois: 0.07313969750999037


## Exercício 2 – gradient check da `DenseLayer` (slide 12)

In [4]:
class NoUpdate:  # optimizer "falso"
    def update(self, w, grad):
        return w

mse = lambda Y, P: np.mean((Y - P) ** 2)
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size

np.random.seed(0)
X, Y = np.random.rand(4, 3), np.random.rand(4, 2)
layer = DenseLayer(2, (3,)).initialize(NoUpdate())
out = layer.forward_propagation(X, training=True)
analytic = X.T @ mse_derivative(Y, out)

numeric, eps = np.zeros_like(layer.weights), 1e-6
for i in range(3):
    for j in range(2):
        layer.weights[i, j] += eps
        lp = mse(Y, layer.forward_propagation(X, True))
        layer.weights[i, j] -= 2 * eps
        lm = mse(Y, layer.forward_propagation(X, True))
        layer.weights[i, j] += eps
        numeric[i, j] = (lp - lm) / (2 * eps)

print(np.abs(numeric - analytic).max())  # ~1e-11

1.513400516017782e-11


## Exercício 3 – derivadas das ativações (slide 19)

In [5]:
x, eps = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7]), 1e-6
for act in [SigmoidActivation(), ReLUActivation(), TanhActivation()]:
    num = (act.activation_function(x + eps) - act.activation_function(x - eps)) / (2 * eps)
    print(act.layer_name(), np.abs(num - act.derivative(x)).max())  # ~1e-10

SigmoidActivation 6.56039597646263e-11
ReLUActivation 1.397779669787269e-10
TanhActivation 3.338995746560158e-11


## Exercício 12.2 – testar a camada `Dropout`

In [6]:
np.random.seed(42)
X = np.random.rand(5, 4)
dropout = Dropout(probability=0.5)

out_train = dropout.forward_propagation(X, training=True)
print("input:\n", X.round(2))
print("mask:\n", dropout.mask)
print("output (treino):\n", out_train.round(2))  # ~metade a zero, o resto multiplicado por 2

out_inf = dropout.forward_propagation(X, training=False)
print("inferência igual ao input:", np.array_equal(out_inf, X))

print("backward = erro * mask:\n", dropout.backward_propagation(np.ones_like(X)))
print("parâmetros:", dropout.parameters())

input:
 [[0.37 0.95 0.73 0.6 ]
 [0.16 0.16 0.06 0.87]
 [0.6  0.71 0.02 0.97]
 [0.83 0.21 0.18 0.18]
 [0.3  0.52 0.43 0.29]]
mask:
 [[1 0 0 0]
 [0 1 0 1]
 [1 0 1 0]
 [0 1 1 1]
 [0 0 1 0]]
output (treino):
 [[0.75 0.   0.   0.  ]
 [0.   0.31 0.   1.73]
 [1.2  0.   0.04 0.  ]
 [0.   0.42 0.36 0.37]
 [0.   0.   0.86 0.  ]]
inferência igual ao input: True
backward = erro * mask:
 [[1. 0. 0. 0.]
 [0. 1. 0. 1.]
 [1. 0. 1. 0.]
 [0. 1. 1. 1.]
 [0. 0. 1. 0.]]
parâmetros: 0


## Exercício 13 – `TanhActivation` e `SoftmaxActivation`

In [7]:
x = np.array([[-2.0, 0.0, 2.0]])
print("tanh:", TanhActivation().forward_propagation(x))  # entre -1 e 1

scores = np.array([[1.0, 2.0, 3.0],
                   [1000.0, 1001.0, 1002.0]])  # valores enormes: a versão estável não dá overflow
probs = SoftmaxActivation().forward_propagation(scores)
print("softmax:\n", probs)
print("somas por linha:", probs.sum(axis=1))

tanh: [[-0.96402758  0.          0.96402758]]
softmax:
 [[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]]
somas por linha: [1. 1.]
